In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scanpy as sc

In [ ]:
import yaml

base_path = Path('../..').resolve()
sys.path.append(str(base_path))
from helpers import singlecell_utils

with open(base_path / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

In [ ]:
GENE_H5AD = base_path / cfg['gene_h5ad']
GENE_H5AD

In [ ]:
MERGED_COUNTS_DIR = Path('../singlecell_counts/')
PAS_H5AD = MERGED_COUNTS_DIR / Path(GENE_H5AD).name.replace('.h5ad', '_PAS.h5ad')

# Read gene-level

In [ ]:
ad_gene = sc.read_h5ad(GENE_H5AD)
ad_gene

In [ ]:
df_gene_psb = singlecell_utils.get_pseudobulk(ad_gene, 'class')
df_gene_psb.index.name = 'gene_name'
df_gene_psb

In [ ]:
df_gene_logcpm = np.log2(1e6 * (df_gene_psb + 0.5) / (df_gene_psb.sum() + 1))
df_gene_logcpm.head()

In [ ]:
df_gene_long = df_gene_logcpm.reset_index().melt(id_vars="gene_name", var_name="cell_type", value_name="logCPM")
df_gene_long.head()

# Read PAS-level

In [ ]:
ad_pas = sc.read_h5ad(PAS_H5AD)
ad_pas

In [ ]:
df_pas_psb = singlecell_utils.get_pseudobulk(ad_pas, 'class')
df_pas_psb

In [ ]:
df_pas_info = pd.read_table('/sc/arion/projects/CommonMind/yeon/p/APA/run_SCAPTURE/5_rename_quantified_PAS/pl_ageXclass/PAS_merged_evaluated_with_name_ageXclass.tsv')
df_pas_info

In [ ]:
pas_to_gn = dict(zip(df_pas_info['PAS_name'], df_pas_info['gene_name']))

In [ ]:
df_pas_psb['gene_name'] = df_pas_psb.index.map(pas_to_gn)
df_pas_psb_gene_sum = df_pas_psb.groupby('gene_name').sum()

In [ ]:
df_pas_gene_sum_logcpm = np.log2(1e6 * (df_pas_psb_gene_sum + 0.5) / (df_pas_psb_gene_sum.sum() + 1))
df_pas_gene_sum_logcpm.head()

In [ ]:
df_pas_long = df_pas_gene_sum_logcpm.reset_index().melt(id_vars="gene_name", var_name="cell_type", value_name="logCPM")

In [ ]:
df_merge = pd.merge(df_gene_long, df_pas_long, on=['gene_name', 'cell_type'], suffixes=('_gene', '_PAS'))
df_merge

In [ ]:
from collections import Counter
df_merge = df_merge[df_merge.logCPM_gene > 0].copy()
Counter(df_merge['cell_type'])

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from scipy import stats
import seaborn as sns

In [ ]:
celltypes = sorted(df_gene_psb.columns)

fig, axes = plt.subplots(2, 4, figsize=(12, 6), constrained_layout=True)
axes_flat = axes.flatten()

for ax, ct in zip(axes_flat, celltypes):
    df_merge_ct = df_merge.query('cell_type==@ct')
    corr, _ = stats.pearsonr(df_merge_ct['logCPM_gene'], df_merge_ct['logCPM_PAS'])
    corr_s, _ = stats.spearmanr(df_merge_ct['logCPM_gene'], df_merge_ct['logCPM_PAS'])
    print(corr, corr_s)

    ax.scatter(df_merge_ct['logCPM_PAS'], df_merge_ct['logCPM_gene'],
               alpha=0.2, s=3, edgecolor='none', color='k', rasterized=True, zorder=5)

    ax.axline((0, 0), slope=1, color='red', linewidth=1, linestyle='--', zorder=10)

    ax.xaxis.grid(zorder=0)
    ax.yaxis.grid(zorder=0)
    ax.set_box_aspect(1)

    ax.text(0.02, 0.98, 'r$_{s}$' + f' = {corr_s:.2f}',
        transform=ax.transAxes, ha='left', va='top',
        bbox=dict(boxstyle='square,pad=0.15', facecolor='white', alpha=0.7, edgecolor='none'))

    ax.set_title(ct)

for ax in axes_flat[len(celltypes):]:
    fig.delaxes(ax)

fig.supxlabel('logCPM PAS, summed per gene')
fig.supylabel('logCPM gene')

plt.savefig('logCPM_gene_vs_PAS.pdf', bbox_inches='tight')

plt.show()
plt.close(fig)